# Temporal Fusion Transformer — WTI Liquidity Prediction

Models the relationship between news sentiment, market context, and WTI trading volume
using a Temporal Fusion Transformer (TFT). The model predicts `log_volume` one hour ahead
and produces attention weights that directly answer RQ1 and RQ2.

**Input features:**
- LLM sentiment features per article (sentiment_score, magnitude, event_type, certainty)
- Market context per hour (DXY, VIX, log_return, price_range)
- Temporal covariates (hour of day, day of week, month)

**Outputs:**
- log_volume prediction
- Attention weights over time (RQ2 — lag structure)
- Feature importance by sentiment direction (RQ1 — asymmetry)

### Necessary imports

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import torch
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, TQDMProgressBar
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.metrics import QuantileLoss
from pytorch_forecasting.data import GroupNormalizer
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")

PyTorch version: 2.11.0
MPS available: True


### Load and merge data from database

Loads articles with LLM features, joins with hourly market context, and constructs
the master modeling dataset. Each row represents one hour with aggregated sentiment
features and market conditions.

In [2]:
conn = sqlite3.connect("../01_data/wti_thesis.db")

# Load articles with LLM features and liquidity
df_articles = pd.read_sql("""
    SELECT 
        l.datetime_hour,
        l.log_volume,
        l.price_range,
        l.log_return,
        l.amihud,
        l.assignment_gap,
        f.sentiment_score,
        f.magnitude,
        f.event_type,
        f.certainty,
        f.price_direction,
        f.time_horizon
    FROM liquidity l
    JOIN llm_features f ON l.article_id = f.article_id
    WHERE l.assignment_gap < 2
""", conn)

# Load hourly market context
df_market = pd.read_sql("""
    SELECT datetime_hour, log_volume, price_range, log_return, amihud, dxy, vix
    FROM market_context
    WHERE log_volume > 0
""", conn)

conn.close()

df_articles['datetime_hour'] = pd.to_datetime(df_articles['datetime_hour'], utc=True)
df_market['datetime_hour'] = pd.to_datetime(df_market['datetime_hour'], utc=True)

print(f"Articles with LLM features: {len(df_articles)}")
print(f"Market context hours: {len(df_market)}")
print(f"Date range: {df_market['datetime_hour'].min()} to {df_market['datetime_hour'].max()}")

Articles with LLM features: 9924
Market context hours: 10797
Date range: 2024-05-10 14:00:00+00:00 to 2026-05-08 20:00:00+00:00


### Aggregate news features to hourly level

Multiple articles can be published in the same hour. Aggregates LLM features
to one row per hour using mean sentiment, max magnitude, and article count.
Hours without news receive neutral values (sentiment=0, magnitude=0, n_articles=0).
The full market context series is used as the base — ensuring a continuous
hourly time series required by the TFT.

In [3]:
# Encode categorical features numerically before aggregation
event_type_map = {
    'geopolitical': 0, 'supply': 1, 'demand': 2, 'macro': 3,
    'inventory': 4, 'technical': 5, 'other': 6
}
price_direction_map = {'bearish': -1, 'neutral': 0, 'bullish': 1}
time_horizon_map = {'immediate': 0, 'short_term': 1, 'long_term': 2}

df_articles['event_type_num'] = df_articles['event_type'].map(event_type_map).fillna(6)
df_articles['price_direction_num'] = df_articles['price_direction'].map(price_direction_map).fillna(0)
df_articles['time_horizon_num'] = df_articles['time_horizon'].map(time_horizon_map).fillna(1)

# Aggregate to hourly
df_hourly = df_articles.groupby('datetime_hour').agg(
    n_articles=('sentiment_score', 'count'),
    sentiment_score=('sentiment_score', 'mean'),
    magnitude=('magnitude', 'max'),
    certainty=('certainty', 'mean'),
    event_type_num=('event_type_num', lambda x: x.mode()[0]),
    price_direction_num=('price_direction_num', 'mean'),
    time_horizon_num=('time_horizon_num', lambda x: x.mode()[0]),
).reset_index()

# Merge with full market context (left join keeps all trading hours)
df_model = df_market.merge(df_hourly, on='datetime_hour', how='left')

# Fill hours with no news with neutral values
df_model['n_articles'] = df_model['n_articles'].fillna(0)
df_model['sentiment_score'] = df_model['sentiment_score'].fillna(0)
df_model['magnitude'] = df_model['magnitude'].fillna(0)
df_model['certainty'] = df_model['certainty'].fillna(0)
df_model['event_type_num'] = df_model['event_type_num'].fillna(6)
df_model['price_direction_num'] = df_model['price_direction_num'].fillna(0)
df_model['time_horizon_num'] = df_model['time_horizon_num'].fillna(1)

# Fill missing DXY/VIX with forward fill
df_model['dxy'] = df_model['dxy'].ffill()
df_model['vix'] = df_model['vix'].ffill()

# Drop remaining NaN rows
df_model = df_model.dropna().reset_index(drop=True)

print(f"Hourly modeling dataset: {len(df_model)} rows")
print(f"Hours with news: {(df_model['n_articles'] > 0).sum()}")
print(f"Hours without news: {(df_model['n_articles'] == 0).sum()}")
print(f"\nFeature summary:")
print(df_model[['log_volume', 'sentiment_score', 'magnitude', 'dxy', 'vix']].describe().round(3))

Hourly modeling dataset: 10797 rows
Hours with news: 5167
Hours without news: 5630

Feature summary:
       log_volume  sentiment_score  magnitude        dxy        vix
count   10797.000        10797.000  10797.000  10797.000  10797.000
mean        8.628           -0.020      0.218    101.461     18.440
std         1.524            0.264      0.278      3.513      4.931
min         0.693           -0.850      0.000     95.754     11.550
25%         7.665            0.000      0.000     98.518     15.510
50%         8.791            0.000      0.000    100.040     17.230
75%         9.811            0.000      0.400    104.329     20.120
max        13.305            0.950      0.950    110.054     59.240


### Add temporal covariates and time index

TFT requires a integer time index and benefits from cyclical time features
that capture known market patterns: US session hours (higher volume),
Wednesday EIA release day, and seasonal driving demand.

In [4]:
# Sort by time
df_model = df_model.sort_values('datetime_hour').reset_index(drop=True)

# Integer time index required by TFT
df_model['time_idx'] = range(len(df_model))

# Temporal covariates
df_model['hour'] = df_model['datetime_hour'].dt.hour
df_model['day_of_week'] = df_model['datetime_hour'].dt.dayofweek
df_model['month'] = df_model['datetime_hour'].dt.month
df_model['is_wednesday'] = (df_model['day_of_week'] == 2).astype(int)
df_model['is_us_session'] = (
    (df_model['hour'] >= 13) & (df_model['hour'] <= 21)
).astype(int)

# Group column required by TFT (single asset = single group)
df_model['asset'] = 'WTI'

print(f"Dataset ready for TFT: {len(df_model)} rows")
print(f"Features: {df_model.columns.tolist()}")
print(df_model[['time_idx', 'datetime_hour', 'hour', 'day_of_week', 'is_wednesday', 'is_us_session']].head(5))

Dataset ready for TFT: 10797 rows
Features: ['datetime_hour', 'log_volume', 'price_range', 'log_return', 'amihud', 'dxy', 'vix', 'n_articles', 'sentiment_score', 'magnitude', 'certainty', 'event_type_num', 'price_direction_num', 'time_horizon_num', 'time_idx', 'hour', 'day_of_week', 'month', 'is_wednesday', 'is_us_session', 'asset']
   time_idx             datetime_hour  hour  day_of_week  is_wednesday  \
0         0 2024-05-10 14:00:00+00:00    14            4             0   
1         1 2024-05-10 15:00:00+00:00    15            4             0   
2         2 2024-05-10 16:00:00+00:00    16            4             0   
3         3 2024-05-10 17:00:00+00:00    17            4             0   
4         4 2024-05-10 18:00:00+00:00    18            4             0   

   is_us_session  
0              1  
1              1  
2              1  
3              1  
4              1  


### Train/validation split and TFT dataset definition

Uses the last 20% of the time series as validation — approximately 4 months.
The TFT uses an encoder (past context window) of 48 hours and predicts
1 hour ahead. The encoder length of 48 hours captures two full daily cycles,
allowing the model to learn intraday patterns and the full lag structure
identified in the OLS analysis (peak at lag+6).

In [5]:
# Train/validation split — last 20% as validation
max_time_idx = df_model['time_idx'].max()
val_cutoff = int(max_time_idx * 0.8)

print(f"Total hours: {max_time_idx + 1}")
print(f"Training: time_idx 0 to {val_cutoff} ({val_cutoff + 1} hours)")
print(f"Validation: time_idx {val_cutoff + 1} to {max_time_idx} ({max_time_idx - val_cutoff} hours)")

# Define time-varying features
time_varying_known = ['hour', 'day_of_week', 'month', 'is_wednesday', 'is_us_session']
time_varying_unknown = [
    'log_volume', 'price_range', 'log_return', 'amihud',
    'dxy', 'vix', 'sentiment_score', 'magnitude', 'certainty',
    'n_articles', 'event_type_num', 'price_direction_num', 'time_horizon_num'
]

# Create TFT dataset
max_encoder_length = 48
max_prediction_length = 1

training = TimeSeriesDataSet(
    df_model[df_model['time_idx'] <= val_cutoff],
    time_idx='time_idx',
    target='log_volume',
    group_ids=['asset'],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    time_varying_known_reals=time_varying_known,
    time_varying_unknown_reals=time_varying_unknown,
    target_normalizer=GroupNormalizer(groups=['asset']),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

validation = TimeSeriesDataSet.from_dataset(
    training,
    df_model[df_model['time_idx'] > val_cutoff],
    predict=False,
    stop_randomization=True
)

print(f"\nTraining samples: {len(training)}")
print(f"Validation samples: {len(validation)}")

Total hours: 10797
Training: time_idx 0 to 8636 (8637 hours)
Validation: time_idx 8637 to 10796 (2160 hours)

Training samples: 8589
Validation samples: 2112


In [6]:
print(f"Validation sequences: {len(validation)}")
# Create dataloaders to confirm
train_dataloader = training.to_dataloader(train=True, batch_size=32, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=32, num_workers=0)

print(f"Training batches: {len(train_dataloader)}")
print(f"Validation batches: {len(val_dataloader)}")

# Check one batch shape
x, y = next(iter(train_dataloader))
print(f"\nInput shape: {x['encoder_cont'].shape}")
print(f"Target shape: {y[0].shape}")

Validation sequences: 2112
Training batches: 268
Validation batches: 66

Input shape: torch.Size([32, 48, 22])
Target shape: torch.Size([32, 1])


### Define and initialize the TFT model

The Temporal Fusion Transformer uses:
- **hidden_size:** internal representation dimension — larger = more capacity but slower
- **attention_head_size:** number of parallel attention heads
- **dropout:** regularization to prevent overfitting
- **hidden_continuous_size:** dimension for continuous variable processing
- **loss:** QuantileLoss predicts a distribution rather than a point estimate,
  providing uncertainty quantification alongside the prediction

In [7]:
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=1e-3,
    hidden_size=32,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=16,
    loss=QuantileLoss(),
    log_interval=10,
    reduce_on_plateau_patience=3,
)

print(f"Model parameters: {sum(p.numel() for p in tft.parameters()):,}")
print(f"trainable parameters: {sum(p.numel() for p in tft.parameters() if p.requires_grad):,}")

Model parameters: 112,685
trainable parameters: 112,685


### Train the model

Uses PyTorch Lightning for training with early stopping and learning rate scheduling.
Training on MPS (Apple Silicon) accelerator. Early stopping monitors validation loss
and stops if no improvement after 5 epochs to prevent overfitting.

In [8]:
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, TQDMProgressBar

early_stop = EarlyStopping(
    monitor='val_loss',
    min_delta=1e-4,
    patience=5,
    verbose=True,
    mode='min'
)

lr_monitor = LearningRateMonitor()

progress_bar = TQDMProgressBar(refresh_rate=1)

trainer = L.Trainer(
    max_epochs=30,
    accelerator='mps',
    devices=1,
    gradient_clip_val=0.1,
    callbacks=[early_stop, lr_monitor, progress_bar],
    enable_progress_bar=True,
    enable_model_summary=False,
    log_every_n_steps=10,
)

trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_loss improved. New best score: 0.279


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_loss did not improve in the last 5 records. Best score: 0.210. Signaling Trainer to stop.


### Load trained model
Used Google Colab for training, loading for local inference.

In [ ]:
# Load trained model
tft_loaded = TemporalFusionTransformer.load_from_checkpoint(
    "../models/tft_wti.ckpt",
    map_location='mps'
)
print("Model loaded successfully")